[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/31_gradient_accumulation.ipynb)

# 🟢 Easy: Gradient Accumulation

Implement a **training step with gradient accumulation** — simulating large batches with limited memory.

### Signature
```python
def accumulated_step(model, optimizer, loss_fn, micro_batches) -> float:
    # micro_batches: list of (input, target) tuples
    # Returns: average loss (float)
```

### Algorithm
1. `optimizer.zero_grad()`
2. For each `(x, y)` in micro_batches: `loss = loss_fn(model(x), y) / len(micro_batches)`, then `loss.backward()`
3. `optimizer.step()`
4. Return total accumulated loss

The key insight: dividing each loss by `n` before backward makes accumulated gradients equal to a single large-batch gradient.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [11]:
# ✏️ YOUR IMPLEMENTATION HERE

def accumulated_step(model, optimizer, loss_fn, micro_batches):
  total_loss=0
  optimizer.zero_grad()
  for i in range(len(micro_batches)):
    ip=micro_batches[i][0]
    op=micro_batches[i][1]
    loss=loss_fn(model(ip), op)/ len(micro_batches)
    loss.backward()
    total_loss+=loss
  optimizer.step()
  return total_loss.item()
         # zero_grad, loop (forward, scale loss, backward), step

In [12]:
# 🧪 Debug
model = nn.Linear(4, 2)
opt = torch.optim.SGD(model.parameters(), lr=0.01)
loss = accumulated_step(model, opt, nn.MSELoss(),
    [(torch.randn(2, 4), torch.randn(2, 2)) for _ in range(4)])
print('Loss:', loss)

Loss: 1.4106682538986206


In [13]:
for _ in range(4):
  print(torch.randn(2, 4), torch.randn(2, 2))

tensor([[-0.2964,  0.8281, -0.0820, -0.3399],
        [ 0.2621, -0.2115,  0.6720, -0.5328]]) tensor([[-0.1896, -0.1901],
        [ 1.3565,  0.7341]])
tensor([[ 0.8062,  0.7410,  0.6088, -0.5560],
        [ 0.5477,  0.4383,  0.0394, -1.1895]]) tensor([[-0.5015, -0.1455],
        [ 0.1522, -1.4437]])
tensor([[-0.2284, -0.3552, -0.7219, -0.3061],
        [ 1.3768,  1.0055, -0.2455, -0.2918]]) tensor([[ 1.5491, -0.8243],
        [-0.6664, -1.1431]])
tensor([[-0.1809, -0.9764, -2.1870, -0.2529],
        [-0.5116, -0.7427,  1.0541, -1.3517]]) tensor([[-1.1933,  1.6208],
        [-1.2068, -2.8007]])


In [14]:
# ✅ SUBMIT
from torch_judge import check
check('gradient_accumulation')


🧪 Testing: Gradient Accumulation (Easy)
──────────────────────────────────────────────────
  ✅ [1/3] Matches full batch update (7.3ms)
  ✅ [2/3] Returns loss value (1.4ms)
  ✅ [3/3] Parameters actually update (1.0ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (9.8ms total)
  Progress saved. Run status() to see your dashboard.

